In [ ]:
!pip install catboost lightgbm optuna polars shap

In [ ]:
import os
os.environ["CB_TASK_TYPE"] = "GPU"

import gc
import sys
import math
import warnings
import numpy as np
import pandas as pd
import polars as pl
import catboost as cb
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

# 1. Verilerin İndirilmesi, Okunması ve Feature Engineering

## 1.1 Kaggle API ile Verinin İndirilmesi

Bu kısımda, takımımız Google Colab'de çalışma yaptığı için veriyi Colab'e indirdik. Buradaki KAGGLE_USERNAME ve KAGGLE_KEY değişkenleri için https://www.kaggle.com/settings adresindeki API kısmınden "Create New Token" butonuyla yeni API key oluşturmalı ve `kaggle.json` dosyasını Google Colab'e veya çalışılan ortama upload etmelisiniz. Değişkenleri ise `kaggle.json` dosyasının içinden bularak ayarlayabilirsiniz.

In [ ]:
!export KAGGLE_USERNAME=xxxxxxxxxxxxxxxx
!export KAGGLE_KEY=xxxxxxxxxxxxxxxx

In [ ]:
!mv /content/kaggle.json /root/.config/kaggle/kaggle.json # Bu satır bazen ilk seferde hata verebiliyor, ikinci kez çalıştırmanız gerekebilir!

!kaggle competitions download -c trendyol-e-ticaret-hackathonu-2025-kaggle

!unzip /content/trendyol-e-ticaret-hackathonu-2025-kaggle.zip -d /content/

In [ ]:
DATA_PATH = "/content/data"

sys.path.append(os.path.join(DATA_PATH))
from trendyol_metric_group_auc import score

## 1.2 Verilerin Okunması

In [ ]:
train_sessions = pl.read_parquet(f"{DATA_PATH}/train_sessions.parquet")
test_sessions = pl.read_parquet(f"{DATA_PATH}/test_sessions.parquet")
content_metadata = pl.read_parquet(f"{DATA_PATH}/content/metadata.parquet")
content_price_data = pl.read_parquet(f"{DATA_PATH}/content/price_rate_review_data.parquet")
user_metadata = pl.read_parquet(f"{DATA_PATH}/user/metadata.parquet")

content_search_log = pl.read_parquet(f"{DATA_PATH}/content/search_log.parquet")
content_sitewide_log = pl.read_parquet(f"{DATA_PATH}/content/sitewide_log.parquet")
user_search_log = pl.read_parquet(f"{DATA_PATH}/user/search_log.parquet")
user_sitewide_log = pl.read_parquet(f"{DATA_PATH}/user/sitewide_log.parquet")
term_search_log = pl.read_parquet(f"{DATA_PATH}/term/search_log.parquet")
content_top_terms_log = pl.read_parquet(f"{DATA_PATH}/content/top_terms_log.parquet")
user_fashion_search_log = pl.read_parquet(f"{DATA_PATH}/user/fashion_search_log.parquet")
user_fashion_sitewide_log = pl.read_parquet(f"{DATA_PATH}/user/fashion_sitewide_log.parquet")
user_top_terms_log = pl.read_parquet(f"{DATA_PATH}/user/top_terms_log.parquet")

## 1.3 Zaman Damgalarının İşlenmesi

In [ ]:
train_sessions = train_sessions.with_columns([
    train_sessions["ts_hour"].cast(pl.Date).alias("ts_date"),
    train_sessions["ts_hour"].dt.hour().alias("hour_of_day"),
    train_sessions["ts_hour"].dt.weekday().alias("day_of_week"),
    train_sessions["ts_hour"].dt.month().alias("month"),
    train_sessions["ts_hour"].dt.year().alias("year")
])

test_sessions = test_sessions.with_columns([
    test_sessions["ts_hour"].cast(pl.Date).alias("ts_date"),
    test_sessions["ts_hour"].dt.hour().alias("hour_of_day"),
    test_sessions["ts_hour"].dt.weekday().alias("day_of_week"),
    test_sessions["ts_hour"].dt.month().alias("month"),
    test_sessions["ts_hour"].dt.year().alias("year")
])

## 1.4 İçerik Güncelleme Tarihlerinden Recency Weight Hesaplanması

In [ ]:
HALF_LIFE_DAYS = 14.0
DECAY_ALPHA = math.log(2.0) / HALF_LIFE_DAYS

def _add_recency_weight(df: pl.DataFrame, date_col: str) -> pl.DataFrame:
    if date_col not in df.columns:
        return df
    df = df.with_columns([
        pl.col(date_col).cast(pl.Date).alias(date_col)
    ])
    max_date = df.select(pl.col(date_col).max()).to_series()[0]
    if max_date is None:
        return df
    return df.with_columns([
        (-(pl.lit(max_date) - pl.col(date_col)).dt.total_days().cast(pl.Float64) * pl.lit(DECAY_ALPHA)).exp().alias("recency_weight")
    ])

content_price_data = content_price_data.with_columns(
    content_price_data["update_date"].cast(pl.Date).alias("ts_date")
)

## 1.5 Eğitim ve Test Oturumlarının Çekirdek Metadata ile Zenginleştirilmesi

In [ ]:
train_sessions = train_sessions.join(content_metadata, on="content_id_hashed", how="left")
train_sessions = train_sessions.join(content_price_data, on=["content_id_hashed", "ts_date"], how="left")
train_sessions = train_sessions.join(user_metadata, on="user_id_hashed", how="left")

test_sessions = test_sessions.join(content_metadata, on="content_id_hashed", how="left")
test_sessions = test_sessions.join(content_price_data, on=["content_id_hashed", "ts_date"], how="left")
test_sessions = test_sessions.join(user_metadata, on="user_id_hashed", how="left")

## 1.6 Arama ve Sitewide Loglarından CTR ve CVR Özelliklerinin Türetilmesi

In [ ]:
content_search_agg = content_search_log.group_by("content_id_hashed").agg([
    pl.col("total_search_impression").mean().alias("cnt_search_imp_mean"),
    pl.col("total_search_click").mean().alias("cnt_search_clk_mean"),
    pl.col("total_search_impression").sum().alias("cnt_search_imp_sum"),
    pl.col("total_search_click").sum().alias("cnt_search_clk_sum"),
])
content_search_agg = content_search_agg.with_columns([
    pl.when(pl.col("cnt_search_imp_sum") > 0)
    .then(pl.col("cnt_search_clk_sum") / pl.col("cnt_search_imp_sum"))
    .otherwise(0).alias("cnt_search_ctr_sum"),
    pl.when(pl.col("cnt_search_imp_mean") > 0)
    .then(pl.col("cnt_search_clk_mean") / pl.col("cnt_search_imp_mean"))
    .otherwise(0).alias("cnt_search_ctr_mean")
])

content_search_log_rec = _add_recency_weight(content_search_log, "date")
content_search_agg_recent = content_search_log_rec.group_by("content_id_hashed").agg([
    (pl.col("total_search_impression") * pl.col("recency_weight")).sum().alias("cnt_search_imp_recent"),
    (pl.col("total_search_click") * pl.col("recency_weight")).sum().alias("cnt_search_clk_recent"),
]).with_columns([
    pl.when(pl.col("cnt_search_imp_recent") > 0)
    .then(pl.col("cnt_search_clk_recent") / pl.col("cnt_search_imp_recent"))
    .otherwise(0).alias("cnt_search_ctr_recent")
])

content_site_agg = content_sitewide_log.group_by("content_id_hashed").agg([
    pl.col("total_click").mean().alias("cnt_site_clk_mean"),
    pl.col("total_cart").mean().alias("cnt_site_cart_mean"),
    pl.col("total_fav").mean().alias("cnt_site_fav_mean"),
    pl.col("total_order").mean().alias("cnt_site_ord_mean"),
    pl.col("total_click").sum().alias("cnt_site_clk_sum"),
    pl.col("total_order").sum().alias("cnt_site_ord_sum")
])
content_site_agg = content_site_agg.with_columns([
    pl.when(pl.col("cnt_site_clk_sum") > 0)
    .then(pl.col("cnt_site_ord_sum") / pl.col("cnt_site_clk_sum"))
    .otherwise(0).alias("cnt_site_cvr_sum"),
    pl.when(pl.col("cnt_site_clk_mean") > 0)
    .then(pl.col("cnt_site_ord_mean") / pl.col("cnt_site_clk_mean"))
    .otherwise(0).alias("cnt_site_cvr_mean")
])

content_sitewide_log_rec = _add_recency_weight(content_sitewide_log, "date")
content_site_agg_recent = content_sitewide_log_rec.group_by("content_id_hashed").agg([
    (pl.col("total_click") * pl.col("recency_weight")).sum().alias("cnt_site_clk_recent"),
    (pl.col("total_order") * pl.col("recency_weight")).sum().alias("cnt_site_ord_recent"),
]).with_columns([
    pl.when(pl.col("cnt_site_clk_recent") > 0)
    .then(pl.col("cnt_site_ord_recent") / pl.col("cnt_site_clk_recent"))
    .otherwise(0).alias("cnt_site_cvr_recent")
])

user_search_agg = user_search_log.group_by("user_id_hashed").agg([
    pl.col("total_search_impression").mean().alias("usr_search_imp_mean"),
    pl.col("total_search_click").mean().alias("usr_search_clk_mean"),
    pl.col("total_search_impression").sum().alias("usr_search_imp_sum"),
    pl.col("total_search_click").sum().alias("usr_search_clk_sum")
])
user_search_agg = user_search_agg.with_columns([
    pl.when(pl.col("usr_search_imp_sum") > 0)
    .then(pl.col("usr_search_clk_sum") / pl.col("usr_search_imp_sum"))
    .otherwise(0).alias("usr_search_ctr_sum"),
    pl.when(pl.col("usr_search_imp_mean") > 0)
    .then(pl.col("usr_search_clk_mean") / pl.col("usr_search_imp_mean"))
    .otherwise(0).alias("usr_search_ctr_mean")
])

user_site_agg = user_sitewide_log.group_by("user_id_hashed").agg([
    pl.col("total_click").mean().alias("usr_site_clk_mean"),
    pl.col("total_cart").mean().alias("usr_site_cart_mean"),
    pl.col("total_fav").mean().alias("usr_site_fav_mean"),
    pl.col("total_order").mean().alias("usr_site_ord_mean"),
    pl.col("total_click").sum().alias("usr_site_clk_sum"),
    pl.col("total_order").sum().alias("usr_site_ord_sum")
])
user_site_agg = user_site_agg.with_columns([
    pl.when(pl.col("usr_site_clk_sum") > 0)
    .then(pl.col("usr_site_ord_sum") / pl.col("usr_site_clk_sum"))
    .otherwise(0).alias("usr_site_cvr_sum"),
    pl.when(pl.col("usr_site_clk_mean") > 0)
    .then(pl.col("usr_site_ord_mean") / pl.col("usr_site_clk_mean"))
    .otherwise(0).alias("usr_site_cvr_mean")
])

user_term_agg = user_top_terms_log.group_by(["user_id_hashed","search_term_normalized"]).agg([
    pl.col("total_search_impression").sum().alias("usr_term_imp_sum"),
    pl.col("total_search_click").sum().alias("usr_term_clk_sum"),
    pl.col("total_search_impression").mean().alias("usr_term_imp_mean"),
    pl.col("total_search_click").mean().alias("usr_term_clk_mean")
]).with_columns([
    pl.when(pl.col("usr_term_imp_sum")>0).then(pl.col("usr_term_clk_sum")/pl.col("usr_term_imp_sum")).otherwise(0).alias("usr_term_ctr_sum"),
    pl.when(pl.col("usr_term_imp_mean")>0).then(pl.col("usr_term_clk_mean")/pl.col("usr_term_imp_mean")).otherwise(0).alias("usr_term_ctr_mean")
])

user_top_terms_log_rec = _add_recency_weight(user_top_terms_log, "ts_hour")
user_term_agg_recent = user_top_terms_log_rec.group_by(["user_id_hashed","search_term_normalized"]).agg([
    (pl.col("total_search_impression") * pl.col("recency_weight")).sum().alias("usr_term_imp_recent"),
    (pl.col("total_search_click") * pl.col("recency_weight")).sum().alias("usr_term_clk_recent"),
]).with_columns([
    pl.when(pl.col("usr_term_imp_recent") > 0)
    .then(pl.col("usr_term_clk_recent") / pl.col("usr_term_imp_recent"))
    .otherwise(0).alias("usr_term_ctr_recent")
])

uf_site_with_cat = user_fashion_sitewide_log.join(
    content_metadata.select(["content_id_hashed","level1_category_name"]),
    on="content_id_hashed",
    how="left"
)
user_cat_agg = uf_site_with_cat.group_by(["user_id_hashed","level1_category_name"]).agg([
    pl.col("total_click").sum().alias("usr_cat_clk_sum"),
    pl.col("total_order").sum().alias("usr_cat_ord_sum")
]).with_columns([
    pl.when(pl.col("usr_cat_clk_sum")>0).then(pl.col("usr_cat_ord_sum")/pl.col("usr_cat_clk_sum")).otherwise(0).alias("usr_cat_cvr_sum")
])

term_agg = term_search_log.group_by("search_term_normalized").agg([
    pl.col("total_search_impression").mean().alias("term_imp_mean"),
    pl.col("total_search_click").mean().alias("term_clk_mean"),
    pl.col("total_search_impression").sum().alias("term_imp_sum"),
    pl.col("total_search_click").sum().alias("term_clk_sum")
])
term_agg = term_agg.with_columns([
    pl.when(pl.col("term_imp_sum") > 0)
    .then(pl.col("term_clk_sum") / pl.col("term_imp_sum"))
    .otherwise(0).alias("term_ctr_sum"),
    pl.when(pl.col("term_imp_mean") > 0)
    .then(pl.col("term_clk_mean") / pl.col("term_imp_mean"))
    .otherwise(0).alias("term_ctr_mean")
])

term_search_log_rec = _add_recency_weight(term_search_log, "ts_hour")
term_agg_recent = term_search_log_rec.group_by("search_term_normalized").agg([
    (pl.col("total_search_impression") * pl.col("recency_weight")).sum().alias("term_imp_recent"),
    (pl.col("total_search_click") * pl.col("recency_weight")).sum().alias("term_clk_recent"),
]).with_columns([
    pl.when(pl.col("term_imp_recent") > 0)
    .then(pl.col("term_clk_recent") / pl.col("term_imp_recent"))
    .otherwise(0).alias("term_ctr_recent")
])

content_term_ctr = content_top_terms_log.group_by(["content_id_hashed", "search_term_normalized"]).agg([
    pl.col("total_search_impression").sum().alias("ct_imp_sum"),
    pl.col("total_search_click").sum().alias("ct_clk_sum")
]).with_columns([
    pl.when(pl.col("ct_imp_sum") > 0)
    .then(pl.col("ct_clk_sum") / pl.col("ct_imp_sum"))
    .otherwise(0).alias("ct_ctr_sum")
])

content_top_terms_log_rec = _add_recency_weight(content_top_terms_log, "date")
content_term_ctr_recent = content_top_terms_log_rec.group_by(["content_id_hashed", "search_term_normalized"]).agg([
    (pl.col("total_search_impression") * pl.col("recency_weight")).sum().alias("ct_imp_recent"),
    (pl.col("total_search_click") * pl.col("recency_weight")).sum().alias("ct_clk_recent"),
]).with_columns([
    pl.when(pl.col("ct_imp_recent") > 0)
    .then(pl.col("ct_clk_recent") / pl.col("ct_imp_recent"))
    .otherwise(0).alias("ct_ctr_recent")
])

uf_search = user_fashion_search_log.group_by(["user_id_hashed", "content_id_hashed"]).agg([
    pl.col("total_search_impression").sum().alias("uf_imp_sum"),
    pl.col("total_search_click").sum().alias("uf_clk_sum")
]).with_columns([
    pl.when(pl.col("uf_imp_sum") > 0)
    .then(pl.col("uf_clk_sum") / pl.col("uf_imp_sum"))
    .otherwise(0).alias("uf_ctr_sum")
])

uf_site = user_fashion_sitewide_log.group_by(["user_id_hashed", "content_id_hashed"]).agg([
    pl.col("total_click").sum().alias("uf_site_clk_sum"),
    pl.col("total_order").sum().alias("uf_site_ord_sum")
]).with_columns([
    pl.when(pl.col("uf_site_clk_sum") > 0)
    .then(pl.col("uf_site_ord_sum") / pl.col("uf_site_clk_sum"))
    .otherwise(0).alias("uf_site_cvr_sum")
])

## 1.7 CV Tags Tokenizasyonu ve Term–IDF Tabanlı Eşleşme Özellikleri

In [ ]:
cv_tags_terms = (
    content_metadata
    .select(["content_id_hashed","cv_tags"])
    .with_columns([
        pl.col("cv_tags").fill_null("").str.to_lowercase()
        .str.replace_all(r"[^a-z0-9]+"," ")
        .str.strip_chars()
        .str.split(" ").alias("cv_tag_tokens")
    ])
    .explode("cv_tag_tokens")
    .with_columns([
        pl.col("cv_tag_tokens").str.strip_chars().alias("cv_tag_term")
    ])
    .filter((pl.col("cv_tag_term").is_not_null()) & (pl.col("cv_tag_term")!=""))
    .select(["content_id_hashed", pl.col("cv_tag_term").alias("search_term_normalized")])
    .unique()
)
term_df_counts = cv_tags_terms.group_by("search_term_normalized").agg([
    pl.count().alias("term_df")
])
num_contents_total = int(content_metadata.select(pl.col("content_id_hashed").n_unique()).to_series()[0])
term_idf = term_df_counts.with_columns([
    ((pl.lit(num_contents_total + 1)).cast(pl.Float64).log() - (pl.col("term_df") + 1).cast(pl.Float64).log()).alias("term_idf")
])
cv_tags_match = cv_tags_terms.join(term_idf, on="search_term_normalized", how="left").with_columns([
    pl.lit(1.0).alias("term_in_cv_tags"),
    pl.col("term_idf").fill_null(0.0).alias("term_in_cv_tags_idf")
])

## 1.8 Eğitim ve Test Setlerine Özellik Agregasyonlarının Join Edilmesi

In [ ]:
for df_name in ["train_sessions", "test_sessions"]:
    df = locals()[df_name]
    df = df.join(content_search_agg, on="content_id_hashed", how="left")
    df = df.join(content_search_agg_recent, on="content_id_hashed", how="left")
    df = df.join(content_site_agg, on="content_id_hashed", how="left")
    df = df.join(content_site_agg_recent, on="content_id_hashed", how="left")
    df = df.join(user_search_agg, on="user_id_hashed", how="left")
    df = df.join(user_site_agg, on="user_id_hashed", how="left")
    df = df.join(term_agg, on="search_term_normalized", how="left")
    df = df.join(term_agg_recent, on="search_term_normalized", how="left")
    df = df.join(content_term_ctr, on=["content_id_hashed", "search_term_normalized"], how="left")
    df = df.join(content_term_ctr_recent, on=["content_id_hashed", "search_term_normalized"], how="left")
    df = df.join(uf_search, on=["user_id_hashed", "content_id_hashed"], how="left")
    df = df.join(uf_site, on=["user_id_hashed", "content_id_hashed"], how="left")
    df = df.join(user_term_agg, on=["user_id_hashed","search_term_normalized"], how="left")
    df = df.join(user_term_agg_recent, on=["user_id_hashed","search_term_normalized"], how="left")
    df = df.join(cv_tags_match, on=["content_id_hashed","search_term_normalized"], how="left")
    if "level1_category_name" in df.columns:
        df = df.join(user_cat_agg, on=["user_id_hashed","level1_category_name"], how="left")
    locals()[df_name] = df

## 1.9 Null Değerlerin Doldurulması ve Türetilmiş Özelliklerin Oluşturulması

In [ ]:
def fill_nulls(df: pl.DataFrame) -> pl.DataFrame:
    cols_to_zero = [
        "selling_price","original_price","discounted_price","content_review_count","content_review_wth_media_count",
        "content_rate_count","content_rate_avg","attribute_type_count","total_attribute_option_count","merchant_count",
        "filterable_label_count","avg_search_impression","avg_search_click","avg_sitewide_click","avg_sitewide_cart",
        "avg_sitewide_fav","avg_sitewide_order","user_avg_search_impression","user_avg_search_click","user_avg_sitewide_click",
        "user_avg_sitewide_cart","user_avg_sitewide_fav","user_avg_sitewide_order","cnt_search_imp_mean","cnt_search_clk_mean",
        "cnt_search_imp_sum","cnt_search_clk_sum","cnt_search_ctr_sum","cnt_search_ctr_mean","cnt_site_clk_mean",
        "cnt_site_cart_mean","cnt_site_fav_mean","cnt_site_ord_mean","cnt_site_clk_sum","cnt_site_ord_sum","cnt_site_cvr_sum",
        "cnt_site_cvr_mean","usr_search_imp_mean","usr_search_clk_mean","usr_search_imp_sum","usr_search_clk_sum",
        "usr_search_ctr_sum","usr_search_ctr_mean","usr_site_clk_mean","usr_site_cart_mean","usr_site_fav_mean",
        "usr_site_ord_mean","usr_site_clk_sum","usr_site_ord_sum","usr_site_cvr_sum","usr_site_cvr_mean","term_imp_mean",
        "term_clk_mean","term_imp_sum","term_clk_sum","term_ctr_sum","term_ctr_mean","ct_imp_sum","ct_clk_sum","ct_ctr_sum",
        "uf_imp_sum","uf_clk_sum","uf_ctr_sum","uf_site_clk_sum","uf_site_ord_sum","uf_site_cvr_sum",
        "usr_term_imp_sum","usr_term_clk_sum","usr_term_imp_mean","usr_term_clk_mean","usr_term_ctr_sum","usr_term_ctr_mean",
        "usr_cat_clk_sum","usr_cat_ord_sum","usr_cat_cvr_sum",
        "cnt_search_imp_recent","cnt_search_clk_recent","cnt_search_ctr_recent",
        "cnt_site_clk_recent","cnt_site_ord_recent","cnt_site_cvr_recent",
        "term_imp_recent","term_clk_recent","term_ctr_recent",
        "ct_imp_recent","ct_clk_recent","ct_ctr_recent",
        "usr_term_imp_recent","usr_term_clk_recent","usr_term_ctr_recent",
        "term_in_cv_tags","term_in_cv_tags_idf"
    ]
    existing = [c for c in cols_to_zero if c in df.columns]
    df = df.with_columns([pl.col(c).fill_null(0) for c in existing])
    more = []
    if "user_birth_year" in df.columns:
        more.append(pl.col("user_birth_year").fill_null(1990))
    if "user_tenure_in_days" in df.columns:
        more.append(pl.col("user_tenure_in_days").fill_null(365))
    if "user_gender" in df.columns:
        more.append(pl.col("user_gender").fill_null("unknown"))
    if more:
        df = df.with_columns(more)
    return df

train_sessions = fill_nulls(train_sessions)
test_sessions = fill_nulls(test_sessions)

def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns([
        pl.when(pl.col("original_price") > 0).then((pl.col("original_price") - pl.col("selling_price")) / pl.col("original_price")).otherwise(0).alias("discount_rate"),
        pl.when(pl.col("selling_price") > 0).then(pl.col("original_price") / pl.col("selling_price")).otherwise(1).alias("price_ratio"),
        pl.when(pl.col("content_review_count") > 0).then(pl.col("content_review_wth_media_count") / pl.col("content_review_count")).otherwise(0).alias("media_review_ratio"),
        pl.when(pl.col("content_rate_count") > 0).then(pl.col("content_rate_avg") * pl.col("content_rate_count")).otherwise(0).alias("total_rating_score"),
        (pl.col("attribute_type_count") + pl.col("total_attribute_option_count")).alias("total_attributes"),
        pl.when(pl.col("filterable_label_count") > 0).then(pl.col("merchant_count") / pl.col("filterable_label_count")).otherwise(0).alias("merchant_per_label_ratio"),
        pl.when(pl.col("selling_price") > 0).then(pl.col("discounted_price") / pl.col("selling_price")).otherwise(1).alias("discount_price_ratio"),
        pl.when(pl.col("content_rate_count") > 0).then(pl.col("content_rate_avg") * pl.col("content_rate_count") * pl.col("content_review_count")).otherwise(0).alias("engagement_score"),
        pl.when(pl.col("attribute_type_count") > 0).then(pl.col("total_attribute_option_count") / pl.col("attribute_type_count")).otherwise(0).alias("avg_options_per_attribute"),
        pl.when(pl.col("selling_price") > 0).then(pl.col("original_price") - pl.col("selling_price")).otherwise(0).alias("absolute_discount"),
        pl.when(pl.col("content_review_count") > 0).then(pl.col("content_rate_avg") * pl.col("content_review_count")).otherwise(0).alias("review_rating_score"),
        pl.when(pl.col("selling_price") > 0).then(pl.col("selling_price") * pl.col("content_rate_avg")).otherwise(0).alias("price_rating_score"),
        pl.when(pl.col("user_tenure_in_days") > 0).then(pl.col("user_tenure_in_days") / 365).otherwise(1).alias("user_tenure_years"),
        (2025 - pl.col("user_birth_year")).alias("user_age"),
        pl.when(pl.col("user_gender") == "F").then(1).when(pl.col("user_gender") == "M").then(0).otherwise(0.5).alias("user_gender_encoded"),
        pl.when(pl.col("day_of_week").is_in([5,6])).then(1).otherwise(0).alias("is_weekend"),
        pl.when(pl.col("month").is_in([11,12])).then(1).otherwise(0).alias("is_holiday_season"),
        (pl.col("cnt_search_imp_mean") + pl.col("cnt_site_clk_mean")).alias("content_popularity_score"),
        (pl.col("usr_search_imp_mean") + pl.col("usr_site_clk_mean")).alias("user_activity_score"),
        pl.when(pl.col("term_imp_sum") > 0).then(pl.col("term_clk_sum")/pl.col("term_imp_sum")).otherwise(0).alias("term_ctr"),
        pl.when(pl.col("ct_imp_sum") > 0).then(pl.col("ct_clk_sum")/pl.col("ct_imp_sum")).otherwise(0).alias("content_term_ctr"),
    ])
    if "content_creation_date" in df.columns:
        df = df.with_columns([
            (pl.col("ts_date").cast(pl.Date) - pl.col("content_creation_date").cast(pl.Date)).dt.total_days().alias("content_age_days")
        ])
    return df

train_sessions = create_features(train_sessions)
test_sessions = create_features(test_sessions)

## 1.10 Session-Level Özelliklerin Çıkarılması

In [ ]:
session_stats_train = train_sessions.group_by("session_id").agg([
    pl.count().alias("session_length"),
    pl.col("selling_price").mean().alias("avg_price"),
    pl.col("selling_price").std().alias("price_std"),
    pl.col("selling_price").min().alias("min_price"),
    pl.col("selling_price").max().alias("max_price"),
    pl.col("discount_rate").mean().alias("avg_discount"),
    pl.col("discount_rate").std().alias("discount_std"),
    pl.col("content_review_count").mean().alias("avg_reviews"),
    pl.col("content_review_count").std().alias("review_std"),
    pl.col("content_rate_avg").mean().alias("avg_rating"),
    pl.col("content_rate_avg").std().alias("rating_std"),
    pl.col("total_attributes").mean().alias("avg_attributes"),
    pl.col("engagement_score").mean().alias("avg_engagement"),
    pl.col("absolute_discount").mean().alias("avg_absolute_discount")
])

session_stats_test = test_sessions.group_by("session_id").agg([
    pl.count().alias("session_length"),
    pl.col("selling_price").mean().alias("avg_price"),
    pl.col("selling_price").std().alias("price_std"),
    pl.col("selling_price").min().alias("min_price"),
    pl.col("selling_price").max().alias("max_price"),
    pl.col("discount_rate").mean().alias("avg_discount"),
    pl.col("discount_rate").std().alias("discount_std"),
    pl.col("content_review_count").mean().alias("avg_reviews"),
    pl.col("content_review_count").std().alias("review_std"),
    pl.col("content_rate_avg").mean().alias("avg_rating"),
    pl.col("content_rate_avg").std().alias("rating_std"),
    pl.col("total_attributes").mean().alias("avg_attributes"),
    pl.col("engagement_score").mean().alias("avg_engagement"),
    pl.col("absolute_discount").mean().alias("avg_absolute_discount")
])

sess_div_train = train_sessions.group_by("session_id").agg([
    pl.n_unique("content_id_hashed").alias("session_unique_items"),
    pl.n_unique("level1_category_name").alias("session_unique_cat1"),
    pl.n_unique("leaf_category_name").alias("session_unique_leaf")
])
sess_div_test = test_sessions.group_by("session_id").agg([
    pl.n_unique("content_id_hashed").alias("session_unique_items"),
    pl.n_unique("level1_category_name").alias("session_unique_cat1"),
    pl.n_unique("leaf_category_name").alias("session_unique_leaf")
])

train_sessions = train_sessions.join(session_stats_train, on="session_id", how="left")
test_sessions = test_sessions.join(session_stats_test, on="session_id", how="left")
train_sessions = train_sessions.join(sess_div_train, on="session_id", how="left")
test_sessions = test_sessions.join(sess_div_test, on="session_id", how="left")

## 1.11 Oturum Düzeyinde Oranların Hesaplanması

In [ ]:
train_sessions = train_sessions.with_columns([
    pl.when(pl.col("avg_price") > 0).then(pl.col("selling_price")/pl.col("avg_price")).otherwise(1).alias("price_vs_session_avg"),
    pl.when(pl.col("avg_discount") > 0).then(pl.col("discount_rate")/pl.col("avg_discount")).otherwise(1).alias("discount_vs_session_avg"),
    pl.when(pl.col("avg_reviews") > 0).then(pl.col("content_review_count")/pl.col("avg_reviews")).otherwise(1).alias("reviews_vs_session_avg"),
    pl.when(pl.col("avg_rating") > 0).then(pl.col("content_rate_avg")/pl.col("avg_rating")).otherwise(1).alias("rating_vs_session_avg"),
    pl.when(pl.col("avg_attributes") > 0).then(pl.col("total_attributes")/pl.col("avg_attributes")).otherwise(1).alias("attributes_vs_session_avg"),
    pl.when(pl.col("max_price") > pl.col("min_price")).then((pl.col("selling_price")-pl.col("min_price"))/(pl.col("max_price")-pl.col("min_price"))).otherwise(0.5).alias("price_percentile_in_session"),
    pl.when(pl.col("avg_engagement") > 0).then(pl.col("engagement_score")/pl.col("avg_engagement")).otherwise(1).alias("engagement_vs_session_avg")
])

test_sessions = test_sessions.with_columns([
    pl.when(pl.col("avg_price") > 0).then(pl.col("selling_price")/pl.col("avg_price")).otherwise(1).alias("price_vs_session_avg"),
    pl.when(pl.col("avg_discount") > 0).then(pl.col("discount_rate")/pl.col("avg_discount")).otherwise(1).alias("discount_vs_session_avg"),
    pl.when(pl.col("avg_reviews") > 0).then(pl.col("content_review_count")/pl.col("avg_reviews")).otherwise(1).alias("reviews_vs_session_avg"),
    pl.when(pl.col("avg_rating") > 0).then(pl.col("content_rate_avg")/pl.col("avg_rating")).otherwise(1).alias("rating_vs_session_avg"),
    pl.when(pl.col("avg_attributes") > 0).then(pl.col("total_attributes")/pl.col("avg_attributes")).otherwise(1).alias("attributes_vs_session_avg"),
    pl.when(pl.col("max_price") > pl.col("min_price")).then((pl.col("selling_price")-pl.col("min_price"))/(pl.col("max_price")-pl.col("min_price"))).otherwise(0.5).alias("price_percentile_in_session"),
    pl.when(pl.col("avg_engagement") > 0).then(pl.col("engagement_score")/pl.col("avg_engagement")).otherwise(1).alias("engagement_vs_session_avg")
])

train_sessions = train_sessions.with_columns([
    pl.when(pl.col("session_length") > 0).then(pl.col("session_unique_items")/pl.col("session_length")).otherwise(0).alias("unique_item_ratio_in_session"),
    pl.when(pl.col("session_length") > 0).then(pl.col("session_unique_cat1")/pl.col("session_length")).otherwise(0).alias("unique_cat1_ratio_in_session"),
])
test_sessions = test_sessions.with_columns([
    pl.when(pl.col("session_length") > 0).then(pl.col("session_unique_items")/pl.col("session_length")).otherwise(0).alias("unique_item_ratio_in_session"),
    pl.when(pl.col("session_length") > 0).then(pl.col("session_unique_cat1")/pl.col("session_length")).otherwise(0).alias("unique_cat1_ratio_in_session"),
])

## 1.12 CTR ve CVR Feature'ları

In [ ]:
for df_name in ["train_sessions","test_sessions"]:
    df = locals()[df_name]
    df = df.with_columns([
        ((pl.col("cnt_search_clk_sum") + pl.lit(0.5)) / (pl.col("cnt_search_imp_sum") + pl.lit(1.0))).alias("cnt_search_ctr_sum_sm"),
        ((pl.col("term_clk_sum") + pl.lit(0.5)) / (pl.col("term_imp_sum") + pl.lit(1.0))).alias("term_ctr_sum_sm"),
        ((pl.col("ct_clk_sum") + pl.lit(0.5)) / (pl.col("ct_imp_sum") + pl.lit(1.0))).alias("ct_ctr_sum_sm"),
        ((pl.col("usr_term_clk_sum") + pl.lit(0.5)) / (pl.col("usr_term_imp_sum") + pl.lit(1.0))).alias("usr_term_ctr_sum_sm"),
        ((pl.col("usr_site_ord_sum") + pl.lit(0.5)) / (pl.col("usr_site_clk_sum") + pl.lit(1.0))).alias("usr_site_cvr_sum_sm"),
        ((pl.col("usr_cat_ord_sum") + pl.lit(0.5)) / (pl.col("usr_cat_clk_sum") + pl.lit(1.0))).alias("usr_cat_cvr_sum_sm"),
        ((pl.col("uf_site_ord_sum") + pl.lit(0.5)) / (pl.col("uf_site_clk_sum") + pl.lit(1.0))).alias("uf_site_cvr_sum_sm")
    ])
    locals()[df_name] = df

## 1.13 Kategori Bazlı İstatistikler ve Oran Feature'larının Oluşturulması

In [ ]:
cat1_stats = train_sessions.group_by("level1_category_name").agg([
    pl.col("selling_price").mean().alias("cat1_avg_price"),
    pl.col("selling_price").std().alias("cat1_std_price"),
    pl.col("discount_rate").mean().alias("cat1_avg_discount"),
    pl.col("discount_rate").std().alias("cat1_std_discount"),
    pl.col("content_review_count").mean().alias("cat1_avg_reviews"),
    pl.col("content_review_count").std().alias("cat1_std_reviews"),
    pl.col("content_rate_avg").mean().alias("cat1_avg_rating"),
    pl.col("content_rate_avg").std().alias("cat1_std_rating"),
    pl.col("engagement_score").mean().alias("cat1_avg_engagement"),
    pl.col("engagement_score").std().alias("cat1_std_engagement")
])

leaf_stats = train_sessions.group_by("leaf_category_name").agg([
    pl.col("selling_price").mean().alias("leaf_avg_price"),
    pl.col("selling_price").std().alias("leaf_std_price"),
    pl.col("discount_rate").mean().alias("leaf_avg_discount"),
    pl.col("discount_rate").std().alias("leaf_std_discount"),
    pl.col("content_review_count").mean().alias("leaf_avg_reviews"),
    pl.col("content_review_count").std().alias("leaf_std_reviews"),
    pl.col("content_rate_avg").mean().alias("leaf_avg_rating"),
    pl.col("content_rate_avg").std().alias("leaf_std_rating")
])

for df_name in ["train_sessions","test_sessions"]:
    df = locals()[df_name]
    if "level1_category_name" in df.columns:
        df = df.join(cat1_stats, on="level1_category_name", how="left")
    if "leaf_category_name" in df.columns:
        df = df.join(leaf_stats, on="leaf_category_name", how="left")
    locals()[df_name] = df

def add_category_ratio_columns(df: pl.DataFrame) -> pl.DataFrame:
    cols = []
    if "cat1_avg_price" in df.columns:
        cols.append(pl.when(pl.col("cat1_avg_price") > 0).then(pl.col("selling_price")/pl.col("cat1_avg_price")).otherwise(1).alias("price_vs_cat1_avg"))
        cols.append(pl.when(pl.col("cat1_avg_discount") > 0).then(pl.col("discount_rate")/pl.col("cat1_avg_discount")).otherwise(1).alias("discount_vs_cat1_avg"))
        cols.append(pl.when(pl.col("cat1_avg_rating") > 0).then(pl.col("content_rate_avg")/pl.col("cat1_avg_rating")).otherwise(1).alias("rating_vs_cat1_avg"))
        cols.append(pl.when(pl.col("cat1_avg_reviews") > 0).then(pl.col("content_review_count")/pl.col("cat1_avg_reviews")).otherwise(1).alias("reviews_vs_cat1_avg"))
        cols.append(pl.when(pl.col("cat1_avg_engagement") > 0).then(pl.col("engagement_score")/pl.col("cat1_avg_engagement")).otherwise(1).alias("engagement_vs_cat1_avg"))
        if all(c in df.columns for c in ["cat1_std_price","cat1_std_discount","cat1_std_reviews","cat1_std_rating","cat1_std_engagement"]):
            cols.extend([
                ((pl.col("selling_price") - pl.col("cat1_avg_price")) / (pl.col("cat1_std_price") + pl.lit(1e-6))).alias("price_z_cat1"),
                ((pl.col("discount_rate") - pl.col("cat1_avg_discount")) / (pl.col("cat1_std_discount") + pl.lit(1e-6))).alias("discount_z_cat1"),
                ((pl.col("content_rate_avg") - pl.col("cat1_avg_rating")) / (pl.col("cat1_std_rating") + pl.lit(1e-6))).alias("rating_z_cat1"),
                ((pl.col("content_review_count") - pl.col("cat1_avg_reviews")) / (pl.col("cat1_std_reviews") + pl.lit(1e-6))).alias("reviews_z_cat1"),
                ((pl.col("engagement_score") - pl.col("cat1_avg_engagement")) / (pl.col("cat1_std_engagement") + pl.lit(1e-6))).alias("engagement_z_cat1"),
            ])
    if "leaf_avg_price" in df.columns:
        cols.append(pl.when(pl.col("leaf_avg_price") > 0).then(pl.col("selling_price")/pl.col("leaf_avg_price")).otherwise(1).alias("price_vs_leaf_avg"))
        cols.append(pl.when(pl.col("leaf_avg_discount") > 0).then(pl.col("discount_rate")/pl.col("leaf_avg_discount")).otherwise(1).alias("discount_vs_leaf_avg"))
        cols.append(pl.when(pl.col("leaf_avg_rating") > 0).then(pl.col("content_rate_avg")/pl.col("leaf_avg_rating")).otherwise(1).alias("rating_vs_leaf_avg"))
        cols.append(pl.when(pl.col("leaf_avg_reviews") > 0).then(pl.col("content_review_count")/pl.col("leaf_avg_reviews")).otherwise(1).alias("reviews_vs_leaf_avg"))
        if all(c in df.columns for c in ["leaf_std_price","leaf_std_discount","leaf_std_reviews","leaf_std_rating"]):
            cols.extend([
                ((pl.col("selling_price") - pl.col("leaf_avg_price")) / (pl.col("leaf_std_price") + pl.lit(1e-6))).alias("price_z_leaf"),
                ((pl.col("discount_rate") - pl.col("leaf_avg_discount")) / (pl.col("leaf_std_discount") + pl.lit(1e-6))).alias("discount_z_leaf"),
                ((pl.col("content_rate_avg") - pl.col("leaf_avg_rating")) / (pl.col("leaf_std_rating") + pl.lit(1e-6))).alias("rating_z_leaf"),
                ((pl.col("content_review_count") - pl.col("leaf_avg_reviews")) / (pl.col("leaf_std_reviews") + pl.lit(1e-6))).alias("reviews_z_leaf"),
            ])
    if cols:
        df = df.with_columns(cols)
    return df

train_sessions = add_category_ratio_columns(train_sessions)
test_sessions = add_category_ratio_columns(test_sessions)

for df_name in ["train_sessions","test_sessions"]:
    df = locals()[df_name]
    df = df.with_columns([
        pl.when((pl.col("term_ctr") > 0) & (pl.col("content_term_ctr") > 0))
        .then(pl.col("content_term_ctr")/(pl.col("term_ctr") + pl.lit(1e-6)))
        .otherwise(0).alias("ct_over_term_ctr")
    ])
    locals()[df_name] = df

## 1.14 Oturum-Kategori Payı Özelliğinin Hesaplanması

In [ ]:
sess_cat_train = train_sessions.group_by(["session_id","level1_category_name"]).agg([
    pl.count().alias("session_cat_count")
])
sess_cat_test = test_sessions.group_by(["session_id","level1_category_name"]).agg([
    pl.count().alias("session_cat_count")
])
train_sessions = train_sessions.join(sess_cat_train, on=["session_id","level1_category_name"], how="left")
test_sessions = test_sessions.join(sess_cat_test, on=["session_id","level1_category_name"], how="left")

train_sessions = train_sessions.with_columns([
    pl.when(pl.col("session_length") > 0).then(pl.col("session_cat_count")/pl.col("session_length")).otherwise(0).alias("category_share_in_session")
])
test_sessions = test_sessions.with_columns([
    pl.when(pl.col("session_length") > 0).then(pl.col("session_cat_count")/pl.col("session_length")).otherwise(0).alias("category_share_in_session")
])

## 1.15 İçerik Düzeyinde İstatistikler ve Oran Feature'ları

In [ ]:
content_stats_train = train_sessions.group_by("content_id_hashed").agg([
    pl.count().alias("content_frequency"),
    pl.col("selling_price").mean().alias("content_avg_price"),
    pl.col("discount_rate").mean().alias("content_avg_discount"),
    pl.col("content_review_count").mean().alias("content_avg_reviews"),
    pl.col("content_rate_avg").mean().alias("content_avg_rating"),
    pl.col("engagement_score").mean().alias("content_avg_engagement")
])

content_stats_test = test_sessions.group_by("content_id_hashed").agg([
    pl.count().alias("content_frequency"),
    pl.col("selling_price").mean().alias("content_avg_price"),
    pl.col("discount_rate").mean().alias("content_avg_discount"),
    pl.col("content_review_count").mean().alias("content_avg_reviews"),
    pl.col("content_rate_avg").mean().alias("content_avg_rating"),
    pl.col("engagement_score").mean().alias("content_avg_engagement")
])

train_sessions = train_sessions.join(content_stats_train, on="content_id_hashed", how="left")
test_sessions = test_sessions.join(content_stats_test, on="content_id_hashed", how="left")

train_sessions = train_sessions.with_columns([
    pl.when(pl.col("content_avg_price") > 0).then(pl.col("selling_price")/pl.col("content_avg_price")).otherwise(1).alias("price_vs_content_avg"),
    pl.when(pl.col("content_avg_discount") > 0).then(pl.col("discount_rate")/pl.col("content_avg_discount")).otherwise(1).alias("discount_vs_content_avg"),
    pl.when(pl.col("content_avg_engagement") > 0).then(pl.col("engagement_score")/pl.col("content_avg_engagement")).otherwise(1).alias("engagement_vs_content_avg")
])

test_sessions = test_sessions.with_columns([
    pl.when(pl.col("content_avg_price") > 0).then(pl.col("selling_price")/pl.col("content_avg_price")).otherwise(1).alias("price_vs_content_avg"),
    pl.when(pl.col("content_avg_discount") > 0).then(pl.col("discount_rate")/pl.col("content_avg_discount")).otherwise(1).alias("discount_vs_content_avg"),
    pl.when(pl.col("content_avg_engagement") > 0).then(pl.col("engagement_score")/pl.col("content_avg_engagement")).otherwise(1).alias("engagement_vs_content_avg")
])

## 1.16 Nihai Feature Set ve Model Hazırlığı

In [ ]:
cat_cols = ["user_id_hashed","content_id_hashed","search_term_normalized"]
extra_cat = ["level1_category_name","level2_category_name","leaf_category_name","cv_tags"]
cat_cols += [c for c in extra_cat if c in train_sessions.columns]

feature_cols = [
    "content_review_count","content_rate_avg","attribute_type_count","total_attribute_option_count","original_price","selling_price",
    "discount_rate","price_ratio","total_attributes","engagement_score","absolute_discount","review_rating_score","price_rating_score",
    "user_age","user_gender_encoded","user_tenure_years","session_length","avg_price","price_std","min_price","max_price","avg_discount",
    "discount_std","avg_reviews","review_std","avg_rating","rating_std","avg_attributes","avg_engagement","avg_absolute_discount",
    "price_vs_session_avg","discount_vs_session_avg","reviews_vs_session_avg","rating_vs_session_avg","attributes_vs_session_avg",
    "price_percentile_in_session","engagement_vs_session_avg","content_frequency","content_avg_price","content_avg_discount","content_avg_reviews",
    "content_avg_rating","content_avg_engagement","price_vs_content_avg","discount_vs_content_avg","engagement_vs_content_avg",
    "cnt_search_imp_mean","cnt_search_clk_mean","cnt_search_imp_sum","cnt_search_clk_sum","cnt_search_ctr_sum","cnt_search_ctr_mean",
    "cnt_site_clk_mean","cnt_site_cart_mean","cnt_site_fav_mean","cnt_site_ord_mean","cnt_site_clk_sum","cnt_site_ord_sum","cnt_site_cvr_sum",
    "cnt_site_cvr_mean","usr_search_imp_mean","usr_search_clk_mean","usr_search_imp_sum","usr_search_clk_sum","usr_search_ctr_sum","usr_search_ctr_mean",
    "usr_site_clk_mean","usr_site_cart_mean","usr_site_fav_mean","usr_site_ord_mean","usr_site_clk_sum","usr_site_ord_sum","usr_site_cvr_sum",
    "usr_site_cvr_mean","term_imp_mean","term_clk_mean","term_imp_sum","term_clk_sum","term_ctr_sum","term_ctr_mean","term_ctr","ct_imp_sum",
    "ct_clk_sum","ct_ctr_sum","content_term_ctr","uf_imp_sum","uf_clk_sum","uf_ctr_sum","uf_site_clk_sum","uf_site_ord_sum","uf_site_cvr_sum",
    "discount_price_ratio","avg_options_per_attribute","media_review_ratio","merchant_per_label_ratio","hour_of_day","day_of_week","month","year",
    "is_weekend","is_holiday_season","content_popularity_score","user_activity_score",
    "cat1_avg_price","cat1_avg_discount","cat1_avg_reviews","cat1_avg_rating","cat1_avg_engagement",
    "leaf_avg_price","leaf_avg_discount","leaf_avg_reviews","leaf_avg_rating",
    "price_vs_cat1_avg","discount_vs_cat1_avg","rating_vs_cat1_avg","reviews_vs_cat1_avg","engagement_vs_cat1_avg",
    "price_vs_leaf_avg","discount_vs_leaf_avg","rating_vs_leaf_avg","reviews_vs_leaf_avg",
    "category_share_in_session",
    "cat1_std_price","cat1_std_discount","cat1_std_reviews","cat1_std_rating","cat1_std_engagement",
    "leaf_std_price","leaf_std_discount","leaf_std_reviews","leaf_std_rating",
    "price_z_cat1","discount_z_cat1","rating_z_cat1","reviews_z_cat1","engagement_z_cat1",
    "price_z_leaf","discount_z_leaf","rating_z_leaf","reviews_z_leaf",
    "ct_over_term_ctr",
    "usr_term_imp_sum","usr_term_clk_sum","usr_term_imp_mean","usr_term_clk_mean","usr_term_ctr_sum","usr_term_ctr_mean",
    "usr_cat_clk_sum","usr_cat_ord_sum","usr_cat_cvr_sum",
    "cnt_search_ctr_sum_sm","term_ctr_sum_sm","ct_ctr_sum_sm","usr_term_ctr_sum_sm","usr_site_cvr_sum_sm","usr_cat_cvr_sum_sm","uf_site_cvr_sum_sm",
    "cnt_search_imp_recent","cnt_search_clk_recent","cnt_search_ctr_recent",
    "cnt_site_clk_recent","cnt_site_ord_recent","cnt_site_cvr_recent",
    "term_imp_recent","term_clk_recent","term_ctr_recent",
    "ct_imp_recent","ct_clk_recent","ct_ctr_recent",
    "usr_term_imp_recent","usr_term_clk_recent","usr_term_ctr_recent",
    "term_in_cv_tags","term_in_cv_tags_idf",
    "session_unique_items","session_unique_cat1","session_unique_leaf",
    "unique_item_ratio_in_session","unique_cat1_ratio_in_session"
]
if "content_age_days" in train_sessions.columns:
    feature_cols.append("content_age_days")

cat_cols = [c for c in cat_cols if c in train_sessions.columns]

base_cols = ["session_id","content_id_hashed","ordered","clicked"] + cat_cols + feature_cols
base_cols = [c for c in base_cols if c in train_sessions.columns]
base_cols = list(dict.fromkeys(base_cols))

train_df = train_sessions.select(base_cols).to_pandas()

X_cat = train_df[cat_cols].astype(str)
X_num = train_df[[c for c in feature_cols if c in train_df.columns]]
X_full = pd.concat([X_cat, X_num], axis=1)
y_ordered = train_df["ordered"].astype(int)
y_clicked = train_df["clicked"].astype(int)
groups = train_df["session_id"].astype(str)
content_ids_series = train_df["content_id_hashed"].astype(str)

In [ ]:
print(f"Train shape: {X_full.shape}")

# 2. Model Eğitimi, Cross Validation ve OOF

## 2.1 CatBoost Hiperparametreleri ve Model Config

Aşağıdaki parametreler, Optuna kütüphanesi kullanılarak 10 trial study ile elde edilmiştir.

In [ ]:
params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=1600,
    learning_rate=0.035,
    depth=8,
    l2_leaf_reg=6.0,
    bootstrap_type="Bernoulli",
    subsample=0.8,
    random_seed=42,
    verbose=False,
    auto_class_weights="Balanced",
    task_type=os.environ.get("CB_TASK_TYPE", "GPU")
)

final_params_ordered = {
    'iterations': 1924,
    'learning_rate': 0.024511949333002735,
    'depth': 4,
    'l2_leaf_reg': 4.43662146086424,
    'subsample': 0.7585728482414573,
    'bootstrap_type': 'Bernoulli',
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': False,
    'auto_class_weights': 'Balanced',
    'task_type': 'GPU'
}

final_params_clicked = {
    'iterations': 986,
    'learning_rate': 0.030312248427568612,
    'depth': 7,
    'l2_leaf_reg': 15.560680856131095,
    'subsample': 0.9051922125074343,
    'bootstrap_type': 'Bernoulli',
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': False,
    'auto_class_weights': 'Balanced',
    'task_type': 'GPU'
}

## 2.2 Cross Validation ile Model Eğitimi ve Weight Optimizasyonu

In [ ]:
cv = GroupKFold(n_splits=10)

oof_ordered = np.zeros(len(X_full))
oof_clicked = np.zeros(len(X_full))
weights_per_fold = []
best_iters_ordered = []
best_iters_clicked = []
models_ordered = []
models_clicked = []

fold_idx = 0
for train_index, val_index in cv.split(X_full, y_ordered, groups):
    fold_idx += 1
    print(f"Fold {fold_idx}")
    X_tr, X_va = X_full.iloc[train_index], X_full.iloc[val_index]
    y_tr_ord, y_va_ord = y_ordered.iloc[train_index], y_ordered.iloc[val_index]
    y_tr_clk, y_va_clk = y_clicked.iloc[train_index], y_clicked.iloc[val_index]

    cat_features_idx = [X_full.columns.get_loc(c) for c in cat_cols]

    model_ord = cb.CatBoostClassifier(**final_params_ordered)
    model_ord.fit(X_tr, y_tr_ord, eval_set=(X_va, y_va_ord), early_stopping_rounds=200, cat_features=cat_features_idx)
    best_iters_ordered.append(model_ord.get_best_iteration())
    pred_ord = model_ord.predict_proba(X_va)[:, 1]

    model_clk = cb.CatBoostClassifier(**final_params_clicked)
    model_clk.fit(X_tr, y_tr_clk, eval_set=(X_va, y_va_clk), early_stopping_rounds=200, cat_features=cat_features_idx)
    best_iters_clicked.append(model_clk.get_best_iteration())
    pred_clk = model_clk.predict_proba(X_va)[:, 1]

    oof_ordered[val_index] = pred_ord
    oof_clicked[val_index] = pred_clk

    val_meta = pd.DataFrame({
        "session_id": groups.iloc[val_index].values,
        "content_id_hashed": content_ids_series.iloc[val_index].values,
        "pred_ord": pred_ord,
        "pred_clk": pred_clk,
        "ordered": y_va_ord.values,
        "clicked": y_va_clk.values
    })

    weights = np.arange(0.50, 0.81, 0.02)
    best_w = 0.7
    best_cv = -1
    for w in weights:
        val_meta_sorted = val_meta.copy()
        val_meta_sorted["score"] = w*val_meta_sorted["pred_ord"] + (1-w)*val_meta_sorted["pred_clk"]
        val_meta_sorted = val_meta_sorted.sort_values(["session_id","score"], ascending=[True, False])
        submission = val_meta_sorted.groupby("session_id").agg({"content_id_hashed": lambda x: " ".join(x.astype(str))}).reset_index()
        submission.columns = ["session_id","prediction"]

        sol = []
        for sid, grp in val_meta.groupby("session_id"):
            ordered_items = grp.loc[grp["ordered"]==1, "content_id_hashed"].astype(str).tolist()
            clicked_items = grp.loc[grp["clicked"]==1, "content_id_hashed"].astype(str).tolist()
            all_items = grp["content_id_hashed"].astype(str).tolist()
            sol.append({"session_id": sid, "ordered_items": " ".join(ordered_items), "clicked_items": " ".join(clicked_items), "all_items": " ".join(all_items)})
        sol_df = pd.DataFrame(sol)
        try:
            s = score(sol_df, submission, "session_id")
            if s > best_cv:
                best_cv = s
                best_w = w
        except Exception:
            continue
    print(f"Fold {fold_idx} best w: {best_w:.3f} score: {best_cv:.5f}")
    weights_per_fold.append(best_w)
    models_ordered.append(model_ord)
    models_clicked.append(model_clk)

In [ ]:
# RAM'de gereksiz yer kaplayan, artık kullanılmayacak değişkenleri siliyoruz.

gc.collect()

def get_size(obj):
    return sys.getsizeof(obj) / (1024**3)

for name, obj in list(globals().items()):
    if get_size(obj) > 1:
        print(f"{name}: {get_size(obj):.2f} GB")

del train_df
del X_tr
del X_cat
del X_num

## 2.3 Out-of-Fold (OOF) Tahmin Değerlendirmesi

In [ ]:
oof_meta = pd.DataFrame({
    "session_id": groups.values,
    "content_id_hashed": content_ids_series.values,
    "pred_ord": oof_ordered,
    "pred_clk": oof_clicked,
    "ordered": y_ordered.values,
    "clicked": y_clicked.values
})

final_w = float(np.mean(weights_per_fold)) if len(weights_per_fold)>0 else 0.7
print(f"Average ensemble weight: {final_w}")

val_sorted = oof_meta.copy()
val_sorted["score"] = final_w*val_sorted["pred_ord"] + (1-final_w)*val_sorted["pred_clk"]
val_sorted = val_sorted.sort_values(["session_id","score"], ascending=[True, False])
submission_oof = val_sorted.groupby("session_id").agg({"content_id_hashed": lambda x: " ".join(x.astype(str))}).reset_index()
submission_oof.columns = ["session_id","prediction"]

sol_all = []
for sid, grp in oof_meta.groupby("session_id"):
    ordered_items = grp.loc[grp["ordered"]==1, "content_id_hashed"].astype(str).tolist()
    clicked_items = grp.loc[grp["clicked"]==1, "content_id_hashed"].astype(str).tolist()
    all_items = grp["content_id_hashed"].astype(str).tolist()
    sol_all.append({"session_id": sid, "ordered_items": " ".join(ordered_items), "clicked_items": " ".join(clicked_items), "all_items": " ".join(all_items)})
sol_all_df = pd.DataFrame(sol_all)

try:
    cv_score = score(sol_all_df, submission_oof, "session_id")
    print(f"OOF score: {cv_score}")
except Exception as e:
    print(f"OOF score failed: {e}")

## 2.4 Global Ensemble Ağırlık Optimizasyonu

In [ ]:
best_w_global = 0.7
best_cv_global = -1
for w in np.arange(0.40, 0.91, 0.01):
    tmp = oof_meta.copy()
    tmp["score"] = w*tmp["pred_ord"] + (1-w)*tmp["pred_clk"]
    tmp = tmp.sort_values(["session_id","score"], ascending=[True, False])
    submission_tmp = tmp.groupby("session_id").agg({"content_id_hashed": lambda x: " ".join(x.astype(str))}).reset_index()
    submission_tmp.columns = ["session_id","prediction"]
    try:
        s = score(sol_all_df, submission_tmp, "session_id")
        if s > best_cv_global:
            best_cv_global = s
            best_w_global = float(w)
    except Exception:
        continue

In [ ]:
print(f"Global best w: {best_w_global} score: {best_cv_global}")

## 2.5 Final Model Eğitimi

In [ ]:
cat_features_idx_full = [X_full.columns.get_loc(c) for c in cat_cols]

final_iters_ord = int(np.clip(np.mean(best_iters_ordered) if best_iters_ordered else params["iterations"], 800, 2200))
final_iters_clk = int(np.clip(np.mean(best_iters_clicked) if best_iters_clicked else params["iterations"], 800, 2200))

params_full_ord = {**final_params_ordered, **{"iterations": final_iters_ord}}
params_full_clk = {**final_params_clicked, **{"iterations": final_iters_clk}}

model_final_ord = cb.CatBoostClassifier(**params_full_ord)
model_final_clk = cb.CatBoostClassifier(**params_full_clk)

model_final_ord.fit(X_full, y_ordered, cat_features=cat_features_idx_full)
model_final_clk.fit(X_full, y_clicked, cat_features=cat_features_idx_full)

In [ ]:
for name, obj in list(globals().items()):
    if get_size(obj) > 1:
        print(f"{name}: {get_size(obj):.2f} GB")
del X_full

# 3. Test Feature'larının Hazırlanması ve Submission Üretimi

## 3.1 Test Özelliklerinin Hazırlanması

In [ ]:
keep_test_cols = ["session_id","content_id_hashed"] + cat_cols + feature_cols
keep_test_cols = [c for c in keep_test_cols if c in test_sessions.columns]
keep_test_cols = list(dict.fromkeys(keep_test_cols))

test_df = test_sessions.select(keep_test_cols).to_pandas()

X_test_cat = test_df[cat_cols].astype(str)
X_test_num = test_df[[c for c in feature_cols if c in test_df.columns]]
X_test = pd.concat([X_test_cat, X_test_num], axis=1)

## 3.2 Prediction Yapılması

In [ ]:
ptest_ord = model_final_ord.predict_proba(X_test)[:, 1]
ptest_clk = model_final_clk.predict_proba(X_test)[:, 1]
final_w = float(best_w_global) if best_cv_global > 0 else final_w
ptest = final_w*ptest_ord + (1-final_w)*ptest_clk

## 3.3 Submission Oluşturulması

In [ ]:
sub_df = test_df[["session_id","content_id_hashed" ]].copy()
sub_df["score"] = ptest
sub_df = sub_df.sort_values(["session_id","score"], ascending=[True, False])
submission = sub_df.groupby("session_id").agg({"content_id_hashed": lambda x: " ".join(x.astype(str))}).reset_index()
submission.columns = ["session_id","prediction"]

submission.to_csv("submission.csv", index=False)
print("CatBoost Baseline Takımı Trendyol Hackathon 2025 Kaggle aşaması submission dosyası başarıyla oluşturuldu, tebrikler!")

# 4. Submission Ensemble

Takımımız, Kaggle yarışmalarında sıkça kullanılan yöntemlerden birisi olan submission ensemble yöntemini kullanarak en yüksek skorumuzu elde eden submission dosyasını elde etmiştir.

Ensemble script'i, aşağıdaki cell'de verilmiştir.

In [ ]:
import argparse
import csv
import os
from collections import defaultdict
from typing import Dict, List, Tuple


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Ensemble multiple Kaggle submissions (session_id,prediction) via rank fusion."
    )
    parser.add_argument(
        "--subs",
        nargs="+",
        required=True,
        help="List of submission CSV file paths to ensemble.",
    )
    parser.add_argument(
        "--weights",
        nargs="+",
        type=float,
        default=None,
        help="Optional weights for each submission (same length as --subs). Defaults to equal weights.",
    )
    parser.add_argument(
        "--method",
        choices=["rrf", "borda", "exp", "medrank"],
        default="rrf",
        help="Rank fusion method: rrf (default), borda, exp (exponential decay), medrank (sum of ranks).",
    )
    parser.add_argument(
        "--k",
        type=int,
        default=60,
        help="RRF constant k (used only when method=rrf). Typical values: 10-100.",
    )
    parser.add_argument(
        "--alpha",
        type=float,
        default=0.05,
        help="Exponential decay rate alpha (used only when method=exp).",
    )
    parser.add_argument(
        "--missing_rank_penalty",
        type=int,
        default=50,
        help="Penalty added to list length when item missing (only for medrank).",
    )
    parser.add_argument(
        "--output",
        type=str,
        default="output/ensemble_submission.csv",
        help="Path to write the ensembled submission CSV.",
    )
    return parser.parse_args()


def safe_split_prediction(prediction: str) -> List[str]:
    if prediction is None:
        return []
    prediction = prediction.strip()
    if not prediction:
        return []
    items = prediction.split(" ")

    seen = set()
    unique_items: List[str] = []
    for item in items:
        if item and item not in seen:
            seen.add(item)
            unique_items.append(item)
    return unique_items


def contribution( method: str,rank_index_one_based: int, list_length: int, k: int, alpha: float) -> float:
    if method == "rrf":
        return 1.0 / (k + rank_index_one_based)
    if method == "borda":
        return max(0.0, float(list_length - rank_index_one_based))
    if method == "exp":
        return float(pow(2.718281828, -alpha * (rank_index_one_based - 1)))
    return 0.0


def read_submission(path) -> Tuple[Dict[str, List[str]], int]:
    session_to_items: Dict[str, List[str]] = {}
    num_rows = 0
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        if "session_id" not in reader.fieldnames or "prediction" not in reader.fieldnames:
            raise ValueError(f"Submission {path} must have columns: session_id,prediction")
        for row in reader:
            num_rows += 1
            session_id = row["session_id"]
            items = safe_split_prediction(row["prediction"])
            session_to_items[session_id] = items
    return session_to_items, num_rows


def fuse_rankings(submissions: List[Tuple[Dict[str, List[str]], float]], method: str, k: int, alpha: float, missing_rank_penalty: int) -> Dict[str, List[str]]:
    all_sessions: set = set()
    for sess_map, _ in submissions:
        all_sessions.update(sess_map.keys())

    final_rankings: Dict[str, List[str]] = {}

    if method == "medrank":
        for session_id in all_sessions:
            sum_ranks: Dict[str, float] = defaultdict(float)
            per_list_lengths: List[int] = []
            items_union: set = set()

            for sess_map, weight in submissions:
                items = sess_map.get(session_id, [])
                L = len(items)
                per_list_lengths.append(L)
                items_union.update(items)
                item_to_rank = {item: idx + 1 for idx, item in enumerate(items)}
                for item in items:
                    sum_ranks[item] += weight * float(item_to_rank[item])
                penalty_rank = float(L + missing_rank_penalty)
                for item in items_union:
                    if item not in item_to_rank:
                        sum_ranks[item] += weight * penalty_rank

            ordered = sorted(sum_ranks.items(), key=lambda kv: (kv[1], kv[0]))
            final_rankings[session_id] = [item for item, _ in ordered]

        return final_rankings

    for session_id in all_sessions:
        scores: Dict[str, float] = defaultdict(float)
        items_union: set = set()
        for sess_map, weight in submissions:
            items = sess_map.get(session_id, [])
            L = len(items)
            items_union.update(items)
            for idx, item in enumerate(items):
                r1 = idx + 1
                scores[item] += weight * contribution(method, r1, L, k, alpha)

        ordered = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
        final_rankings[session_id] = [item for item, _ in ordered]

    return final_rankings


def write_submission(path: str, rankings: Dict[str, List[str]]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    session_ids = list(rankings.keys())
    session_ids.sort()
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["session_id", "prediction"])
        for session_id in session_ids:
            items = rankings[session_id]
            writer.writerow([session_id, " ".join(items)])


def main():
    args = parse_args()

    if args.weights is not None and len(args.weights) != len(args.subs):
        raise ValueError("--weights length must match number of --subs")

    weights = args.weights if args.weights is not None else [1.0] * len(args.subs)

    loaded: List[Tuple[Dict[str, List[str]], float]] = []
    total_rows = 0
    for path, w in zip(args.subs, weights):
        sess_map, n = read_submission(path)
        loaded.append((sess_map, float(w)))
        total_rows = max(total_rows, n)
        print(f"Loaded {path}: {len(sess_map)} sessions")

    coverage_sets = [set(m.keys()) for m, _ in loaded]
    inter = set.intersection(*coverage_sets) if coverage_sets else set()
    uni = set.union(*coverage_sets) if coverage_sets else set()
    if len(uni) != len(inter):
        print(
            f"Warning: session_id sets differ across submissions (union={len(uni)}, intersection={len(inter)}). Proceeding with union."
        )
    else:
        print(f"Session_id count: {len(uni)}")

    print(
        f"Fusing with method={args.method}, weights={[round(w,3) for w in weights]}, k={args.k}, alpha={args.alpha}"
    )
    rankings = fuse_rankings(
        submissions=loaded,
        method=args.method,
        k=args.k,
        alpha=args.alpha,
        missing_rank_penalty=args.missing_rank_penalty,
    )

    write_submission(args.output, rankings)
    print(f"Ensembled submission written to: {args.output}")


if __name__ == "__main__":
    main()

Script `argparse` kütüphanesi ile uyumlu çalıştığından, yukarıdaki cell'i `ensemble_submission.py` olarak kaydedip Google Colab'a/çalıştığınız ortama yüklemeniz, ardından aşağıdaki komutları kullanarak çalıştırmanız kolaylık sağlayacaktır.

Aşağıda, farklı ensemble yöntemlerinin kullanımlarına dair kodlar vardır.

Ensemble yapılırken kullanılan dosyalar, bu notebook'un minimal değişikliklerle çalıştırılması sonucunda elde edilmiş dosyalardır. Bu dosyaları aşağıdaki kodu kullanarak indirebilirsiniz (trendyoldatascience ve nlztrk kullanıcılarına yetki verilmiştir):

In [ ]:
!kaggle datasets download -d ahmeterdempamuk/catboost-baseline-trendyolhackathon-ensemble-files
!unzip /content/catboost-baseline-trendyolhackathon-ensemble-files.zip -d /content/

Takımımızın en yüksek skoru aldığı ensemble yöntemi ve kullanımı da aşağıda belirtilmiştir:

## 4.1 RRF with $k=60$ (En Yüksek Skorlu Çözüm, Public LB: 0.70423, Private LB: 0.70590)

In [ ]:
!python3 "ensemble_submissions.py" --subs "/content/submission.csv" "/content/submission (1).csv" "/content/submission(1).csv" "/content/submission(1) copy.csv" "/content/submission(2).csv" "/content/submission_blended.csv" --method rrf --k 60 --output "/content/ensemble_rrf_k60.csv"

Bu çalıştırma sonucunda oluşan `ensemble_rrf_k60.csv` dosyası, nihai ve en yüksek skorlu çözüm dosyamızdır.

## 4.2 RRF with $k=50$

In [ ]:
!python3 "scripts/ensemble_submissions.py" --subs "output/submission.csv" "output/submission (1).csv" "output/submission(1).csv" "output/submission(1) copy.csv" "output/submission(2).csv" "output/submission_blended.csv" --method rrf --k 50 --output "output/ensemble_rrf_k50.csv"

## 4.3 Exp with $\alpha = 0.05$

In [ ]:
!python3 "scripts/ensemble_submissions.py" --subs "output/submission.csv" "output/submission (1).csv" "output/submission(1).csv" "output/submission(1) copy.csv" "output/submission(2).csv" "output/submission_blended.csv" --method exp --alpha 0.05 --output "output/ensemble_exp_a005.csv"

## 4.4 MedRank with Penalty Parameter $\lambda = 50$

In [ ]:
!python3 "scripts/ensemble_submissions.py" --subs "output/submission.csv" "output/submission (1).csv" "output/submission(1).csv" "output/submission(1) copy.csv" "output/submission(2).csv" "output/submission_blended.csv" --method medrank --missing_rank_penalty 50 --output "output/ensemble_medrank.csv"

## 4.5 Weighted RRF with $k=60$

In [ ]:
!python3 "scripts/ensemble_submissions.py" --subs "output/submission.csv" "output/submission (1).csv" "output/submission(1).csv" "output/submission(1) copy.csv" "output/submission(2).csv" "output/submission_blended.csv" --weights 1 1 1 1 1 1.2 --method rrf --k 60 --output "output/ensemble_rrf_k60_w.csv"

# NOT:

Lise öğrencilerinden oluşan CatBoost Baseline ekibi olarak, bu güzel yarışmayı düzenleyen ve bizlere gerçek hayat verisiyle çalışma imkanı sağlayan, 2 hafta gibi bir süreçte rekabetçi ve öğretici bir deneyim yaşatan, başta TEKNOFEST ve Trendyol ekibi olmak üzere emeği geçen herkese yürekten teşekkürlerimizi sunarız.